# Project 3 — Data Preparation Notebook

**Purpose:** Download `ade_corpus_v2`, run EDA, split 80/10/10 by text hash, convert to chat-format JSONL for Qwen2.5-7B-Instruct fine-tuning.

**Runs on:** Kaggle (T4×2) or locally.

**Output files:**
- `/kaggle/working/data/processed/train.jsonl`
- `/kaggle/working/data/processed/val.jsonl`
- `/kaggle/working/data/processed/test.jsonl`
- `/kaggle/working/data/processed/dataset_stats.json`

**Key design decisions (see `docs/Project3_Design_Document.md`):**
- D-01: Use all 3 ade_corpus_v2 configs
- D-02: Split grouped by `md5(text)` hash — NOT row-level (prevents leakage from duplicate sentences)
- D-08: Chat format with Qwen2.5 template; model target is `schema_version + entities[] + relation_status` only
- D-35: `record_id` and `validation` are NOT in training targets — system-injected at inference time

## Cell 1 — Install dependencies

In [4]:
# On Kaggle: internet must be ON in notebook settings
import subprocess, sys

pkgs = ["datasets", "pandas", "scikit-learn", "numpy"]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed.")

All packages installed.


## Cell 2 — Imports and paths

In [5]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from sklearn.model_selection import StratifiedShuffleSplit

# ── Output directory ────────────────────────────────────────────────────────
if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/data/processed")
else:
    OUT_DIR = Path("data/processed")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT_DIR.resolve()}")

SEED = 42

Output directory: /kaggle/working/data/processed


## Cell 3 — Download all 3 ade_corpus_v2 configs

In [6]:
# trust_remote_code removed — dataset is now standard Parquet format on HF Hub
print("Downloading ade_corpus_v2 — classification config...")
ds_cls = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_classification")

print("Downloading ade_corpus_v2 — drug_ade_relation config...")
ds_ade = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_drug_ade_relation")

print("Downloading ade_corpus_v2 — drug_dosage_relation config...")
ds_dos = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_drug_dosage_relation")

df_cls = ds_cls["train"].to_pandas()
df_ade = ds_ade["train"].to_pandas()
df_dos = ds_dos["train"].to_pandas()

print(f"\nClassification rows : {len(df_cls):,}")
print(f"Drug-ADE rows       : {len(df_ade):,}")
print(f"Drug-dosage rows    : {len(df_dos):,}")


Classification rows : 23,516
Drug-ADE rows       : 6,821
Drug-dosage rows    : 279


## Cell 3b — DIAGNOSTIC: inspect actual `indexes` field structure

The `indexes` field from HuggingFace may store `start_char`/`end_char` as **single-item lists** `[52]`
rather than scalars `52` after `.to_pandas()`. This cell confirms the actual format so the
builder functions can handle it correctly.

In [7]:
print("=" * 60)
print("ADE DATASET — column names and indexes structure")
print(f"df_ade columns : {list(df_ade.columns)}")
sample_ade = df_ade.iloc[0]
print(f"\nSample row text    : {sample_ade['text'][:80]}")
print(f"Sample row drug    : {sample_ade['drug']}")
print(f"Sample row effect  : {sample_ade['effect']}")
idx = sample_ade["indexes"]
print(f"\nindexes type       : {type(idx)}")
print(f"indexes value      : {idx}")
if isinstance(idx, dict):
    for key in idx:
        sub = idx[key]
        print(f"  indexes['{key}'] type  : {type(sub)}  value: {sub}")
        if isinstance(sub, dict):
            sc = sub.get('start_char')
            print(f"    start_char type : {type(sc)}  value: {sc}")

print("\n" + "=" * 60)
print("DOSAGE DATASET — column names and indexes structure")
print(f"df_dos columns : {list(df_dos.columns)}")
sample_dos = df_dos.iloc[0]
print(f"\nSample row text    : {sample_dos['text'][:80]}")
print(f"Sample row drug    : {sample_dos['drug']}")
print(f"Sample row dosage  : {sample_dos['dosage']}")
idx_d = sample_dos["indexes"]
print(f"\nindexes type       : {type(idx_d)}")
print(f"indexes value      : {idx_d}")

ADE DATASET — column names and indexes structure
df_ade columns : ['text', 'drug', 'effect', 'indexes']

Sample row text    : Intravenous azithromycin-induced ototoxicity.
Sample row drug    : azithromycin
Sample row effect  : ototoxicity

indexes type       : <class 'dict'>
indexes value      : {'drug': {'start_char': array([12], dtype=int32), 'end_char': array([24], dtype=int32)}, 'effect': {'start_char': array([33], dtype=int32), 'end_char': array([44], dtype=int32)}}
  indexes['drug'] type  : <class 'dict'>  value: {'start_char': array([12], dtype=int32), 'end_char': array([24], dtype=int32)}
    start_char type : <class 'numpy.ndarray'>  value: [12]
  indexes['effect'] type  : <class 'dict'>  value: {'start_char': array([33], dtype=int32), 'end_char': array([44], dtype=int32)}
    start_char type : <class 'numpy.ndarray'>  value: [33]

DOSAGE DATASET — column names and indexes structure
df_dos columns : ['text', 'drug', 'dosage', 'indexes']

Sample row text    : An episode of suba

## Cell 4 — EDA

In [8]:
all_texts = pd.concat([
    df_cls[["text"]],
    df_ade[["text"]],
    df_dos[["text"]],
]).drop_duplicates(subset="text").copy()

all_texts["approx_tokens"] = all_texts["text"].str.split().str.len()

print("=" * 50)
print("TEXT LENGTH (approx tokens, whitespace split)")
print(f"  Unique texts : {len(all_texts):,}")
print(f"  Mean         : {all_texts['approx_tokens'].mean():.1f}")
print(f"  Median (p50) : {all_texts['approx_tokens'].median():.1f}")
print(f"  p95          : {all_texts['approx_tokens'].quantile(0.95):.1f}")
print(f"  Max          : {all_texts['approx_tokens'].max():.0f}")

print("\n" + "=" * 50)
print("CLASS BALANCE — Classification config")
cls_counts = df_cls["label"].value_counts().sort_index()
for label, count in cls_counts.items():
    name = "ADE present (positive)" if label == 1 else "No ADE (negative)"
    print(f"  label={label} ({name}): {count:,} ({count/len(df_cls)*100:.1f}%)")

print("\n" + "=" * 50)
print("DUPLICATE SENTENCES (same text in multiple rows)")
ade_dup = df_ade["text"].duplicated(keep=False).sum()
dos_dup = df_dos["text"].duplicated(keep=False).sum()
print(f"  drug_ade_relation   : {ade_dup:,} rows share text with at least one other row")
print(f"  drug_dosage_relation: {dos_dup:,} rows share text with at least one other row")
print("  → These MUST go to the same split. See text_hash grouping below.")

print("\n" + "=" * 50)
print("FIELD COVERAGE")
print(f"  drug_ade_relation rows (drug+ADE entity pairs)  : {len(df_ade):,}")
print(f"  drug_dosage_relation rows (dosage examples)     : {len(df_dos):,}")
dosage_pct = len(df_dos) / len(df_ade) * 100
print(f"  Dosage coverage vs ADE rows                     : {dosage_pct:.1f}%")
print(f"  ⚠ Dosage field is best-effort (~{int(len(df_dos)*0.8)} train examples after split)")

TEXT LENGTH (approx tokens, whitespace split)
  Unique texts : 20,896
  Mean         : 17.8
  Median (p50) : 17.0
  p95          : 33.0
  Max          : 135

CLASS BALANCE — Classification config
  label=0 (No ADE (negative)): 16,695 (71.0%)
  label=1 (ADE present (positive)): 6,821 (29.0%)

DUPLICATE SENTENCES (same text in multiple rows)
  drug_ade_relation   : 3,956 rows share text with at least one other row
  drug_dosage_relation: 111 rows share text with at least one other row
  → These MUST go to the same split. See text_hash grouping below.

FIELD COVERAGE
  drug_ade_relation rows (drug+ADE entity pairs)  : 6,821
  drug_dosage_relation rows (dosage examples)     : 279
  Dosage coverage vs ADE rows                     : 4.1%
  ⚠ Dosage field is best-effort (~223 train examples after split)


## Cell 5 — Build unified training rows with text_hash

In [9]:
def compute_text_hash(text: str) -> str:
    """MD5 hash of normalized text. Used for split grouping ONLY, not deduplication."""
    normalized = " ".join(text.strip().split())
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()


def _get_span(span_dict: dict) -> tuple:
    """
    Extract (start_char, end_char) from an indexes sub-dict.

    ade_corpus_v2 loaded via HuggingFace .to_pandas() wraps Arrow int32 scalars
    in single-item lists: {"start_char": [52], "end_char": [63]}.
    This helper unpacks both list and scalar formats robustly.
    """
    start = span_dict["start_char"]
    end   = span_dict["end_char"]
    if isinstance(start, (list, tuple, np.ndarray)):
        start = start[0]
    if isinstance(end, (list, tuple, np.ndarray)):
        end = end[0]
    return int(start), int(end)


def build_target_json_ade(row) -> dict:
    """
    Build model target JSON from a drug_ade_relation row.
    Each row: one drug entity + one adverse_event entity.
    relation_status = 'related' (all drug_ade_relation rows are positive).
    """
    text    = row["text"]
    indexes = row["indexes"]

    drug_start, drug_end     = _get_span(indexes["drug"])
    effect_start, effect_end = _get_span(indexes["effect"])

    drug_mention   = text[drug_start:drug_end]
    effect_mention = text[effect_start:effect_end]

    # Fallback: if span extraction yields empty string, use the string column directly
    if not drug_mention.strip():
        drug_mention = str(row["drug"])
        drug_start   = text.find(drug_mention)
        drug_end     = drug_start + len(drug_mention) if drug_start >= 0 else len(drug_mention)
    if not effect_mention.strip():
        effect_mention = str(row["effect"])
        effect_start   = text.find(effect_mention)
        effect_end     = effect_start + len(effect_mention) if effect_start >= 0 else len(effect_mention)

    entities = [
        {
            "entity_type":       "medication",
            "mention":           drug_mention,
            "dosage":            None,
            "linked_medication": None,
            "evidence":          text,
            "source_span":       {"start_char": drug_start, "end_char": drug_end},
        },
        {
            "entity_type":       "adverse_event",
            "mention":           effect_mention,
            "dosage":            None,
            "linked_medication": drug_mention,
            "evidence":          text,
            "source_span":       {"start_char": effect_start, "end_char": effect_end},
        },
    ]

    return {
        "schema_version": "v1",
        "entities":       entities,
        "relation_status": "related",
    }


def build_target_json_dosage(row) -> dict:
    """
    Build model target JSON from a drug_dosage_relation row.
    One medication entity with dosage. No ADE. relation_status = 'none'.
    """
    text    = row["text"]
    indexes = row["indexes"]

    drug_start, drug_end = _get_span(indexes["drug"])
    drug_mention = text[drug_start:drug_end]

    if not drug_mention.strip():
        drug_mention = str(row["drug"])
        drug_start   = text.find(drug_mention)
        drug_end     = drug_start + len(drug_mention) if drug_start >= 0 else len(drug_mention)

    entities = [
        {
            "entity_type":       "medication",
            "mention":           drug_mention,
            "dosage":            str(row["dosage"]) if row["dosage"] else None,
            "linked_medication": None,
            "evidence":          text,
            "source_span":       {"start_char": drug_start, "end_char": drug_end},
        }
    ]

    return {
        "schema_version":  "v1",
        "entities":        entities,
        "relation_status": "none",
    }


def build_target_json_negative(text: str) -> dict:
    """Negative example: no entities, relation_status='not_related'."""
    return {
        "schema_version":  "v1",
        "entities":        [],
        "relation_status": "not_related",
    }


print("Target JSON builders defined.")

Target JSON builders defined.


In [10]:
rows = []
n_ade_errors = 0
n_dos_errors = 0

# 1. Drug-ADE relation rows
print(f"Processing {len(df_ade):,} drug-ADE rows...")
for _, row in df_ade.iterrows():
    try:
        target = build_target_json_ade(row)
        rows.append({
            "text":           row["text"],
            "text_hash":      compute_text_hash(row["text"]),
            "relation_label": "related",
            "target_json":    target,
            "source_config":  "drug_ade_relation",
        })
    except Exception as e:
        n_ade_errors += 1
        if n_ade_errors <= 3:  # Print first 3 errors to diagnose without flooding
            print(f"  [ERROR #{n_ade_errors}] {type(e).__name__}: {e}")
            print(f"    text    : {row['text'][:80]}")
            print(f"    indexes : {row.get('indexes')}")

print(f"  Done: {len(df_ade) - n_ade_errors:,} built, {n_ade_errors:,} skipped")

# 2. Drug-dosage relation rows
print(f"\nProcessing {len(df_dos):,} drug-dosage rows...")
for _, row in df_dos.iterrows():
    try:
        target = build_target_json_dosage(row)
        rows.append({
            "text":           row["text"],
            "text_hash":      compute_text_hash(row["text"]),
            "relation_label": "none",
            "target_json":    target,
            "source_config":  "drug_dosage_relation",
        })
    except Exception as e:
        n_dos_errors += 1
        if n_dos_errors <= 3:
            print(f"  [ERROR #{n_dos_errors}] {type(e).__name__}: {e}")
            print(f"    text    : {row['text'][:80]}")
            print(f"    indexes : {row.get('indexes')}")

print(f"  Done: {len(df_dos) - n_dos_errors:,} built, {n_dos_errors:,} skipped")

# 3. Negative classification rows (label=0)
neg_rows = df_cls[df_cls["label"] == 0]
print(f"\nProcessing {len(neg_rows):,} negative classification rows...")
for _, row in neg_rows.iterrows():
    rows.append({
        "text":           row["text"],
        "text_hash":      compute_text_hash(row["text"]),
        "relation_label": "not_related",
        "target_json":    build_target_json_negative(row["text"]),
        "source_config":  "classification_negative",
    })

df_all = pd.DataFrame(rows)
print(f"\n{'='*50}")
print(f"Total rows: {len(df_all):,}")
print(df_all["relation_label"].value_counts())
print()
print(df_all["source_config"].value_counts())

# ── Hard assertions: training data must contain all three label types ─────────
n_related = (df_all["relation_label"] == "related").sum()
n_none    = (df_all["relation_label"] == "none").sum()
assert n_related > 0, (
    f"FATAL: 0 'related' rows — all {len(df_ade)} drug_ade rows failed. "
    "Check ERROR messages above and the DIAGNOSTIC cell output."
)
assert n_none > 0, (
    f"FATAL: 0 'none' rows — all {len(df_dos)} drug_dosage rows failed. "
    "Check ERROR messages above."
)
print(f"\n✓ related rows   : {n_related:,}")
print(f"✓ none rows      : {n_none:,}")
print(f"✓ not_related rows: {(df_all['relation_label'] == 'not_related').sum():,}")

Processing 6,821 drug-ADE rows...
  [ERROR #1] IndexError: index 0 is out of bounds for axis 0 with size 0
    text    : Patients receiving neutral protamine Hagedorn (NPH) insulin are at increased ris
    indexes : {'drug': {'start_char': array([], dtype=int32), 'end_char': array([], dtype=int32)}, 'effect': {'start_char': array([105], dtype=int32), 'end_char': array([131], dtype=int32)}}
  [ERROR #2] IndexError: index 0 is out of bounds for axis 0 with size 0
    text    : OBJECTIVE: To report a case of linear immunoglobulin (Ig) A bullous dermatosis (
    indexes : {'drug': {'start_char': array([97], dtype=int32), 'end_char': array([108], dtype=int32)}, 'effect': {'start_char': array([], dtype=int32), 'end_char': array([], dtype=int32)}}
  [ERROR #3] IndexError: index 0 is out of bounds for axis 0 with size 0
    text    : Diarrhoea, T-CD4+ lymphopenia and bilateral patchy pulmonary infiltrates develop
    indexes : {'drug': {'start_char': array([147], dtype=int32), 'end_char': arra

## Cell 6 — Grouped 80/10/10 split (stratified by relation_label, grouped by text_hash)

In [11]:
LABEL_PRIORITY = {"related": 0, "not_related": 1, "none": 2}

hash_to_label = (
    df_all.groupby("text_hash")["relation_label"]
    .apply(lambda labels: min(labels, key=lambda l: LABEL_PRIORITY[l]))
    .reset_index()
)
hash_to_label.columns = ["text_hash", "primary_label"]

unique_hashes = hash_to_label["text_hash"].values
strat_labels  = hash_to_label["primary_label"].values

print(f"Unique text_hashes : {len(unique_hashes):,}")
print(pd.Series(strat_labels).value_counts())

# First split: 80% train, 20% temp
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(sss1.split(unique_hashes, strat_labels))

train_hashes = set(unique_hashes[train_idx])
temp_hashes  = unique_hashes[temp_idx]
temp_labels  = strat_labels[temp_idx]

# Second split: 50/50 on the 20% → 10% val, 10% test
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(sss2.split(temp_hashes, temp_labels))

val_hashes  = set(temp_hashes[val_idx])
test_hashes = set(temp_hashes[test_idx])

def assign_split(text_hash):
    if text_hash in train_hashes: return "train"
    if text_hash in val_hashes:   return "val"
    return "test"

df_all["split"] = df_all["text_hash"].apply(assign_split)

assert train_hashes.isdisjoint(val_hashes),  "LEAKAGE: train ∩ val not empty!"
assert train_hashes.isdisjoint(test_hashes), "LEAKAGE: train ∩ test not empty!"
assert val_hashes.isdisjoint(test_hashes),   "LEAKAGE: val ∩ test not empty!"
print("\nNo hash leakage across splits. ✓")

split_counts = df_all["split"].value_counts()
total = len(df_all)
print("\nSplit row counts:")
for s in ["train", "val", "test"]:
    n = split_counts.get(s, 0)
    print(f"  {s:5s}: {n:6,} rows ({n/total*100:.1f}%)")

Unique text_hashes : 20,883
not_related    16625
related         4258
Name: count, dtype: int64

No hash leakage across splits. ✓

Split row counts:
  train: 19,014 rows (80.0%)
  val  :  2,375 rows (10.0%)
  test :  2,369 rows (10.0%)


In [12]:
# Verify label distribution is similar across splits
print("Relation label distribution per split:")
for split in ["train", "val", "test"]:
    sub = df_all[df_all["split"] == split]
    counts = sub["relation_label"].value_counts()
    print(f"\n  {split.upper()} ({len(sub):,} rows)")
    for label in ["related", "not_related", "none"]:
        n = counts.get(label, 0)
        print(f"    {label:12s}: {n:5,} ({n/len(sub)*100:.1f}%)")

# Hard assertion: all three splits must have positive (related) examples
for split in ["train", "val", "test"]:
    sub = df_all[df_all["split"] == split]
    n_pos = (sub["relation_label"] == "related").sum()
    assert n_pos > 0, f"FATAL: {split} split has 0 'related' rows — stratification failed."

print("\n✓ All three splits contain positive (related) examples.")

Relation label distribution per split:

  TRAIN (19,014 rows)
    related     : 5,441 (28.6%)
    not_related : 13,359 (70.3%)
    none        :   214 (1.1%)

  VAL (2,375 rows)
    related     :   678 (28.5%)
    not_related : 1,666 (70.1%)
    none        :    31 (1.3%)

  TEST (2,369 rows)
    related     :   665 (28.1%)
    not_related : 1,670 (70.5%)
    none        :    34 (1.4%)

✓ All three splits contain positive (related) examples.


## Cell 7 — Instruction template (verbatim from Design Doc §11.2)

In [13]:
# This template is IDENTICAL at train time and inference time.
# Do not change without updating docs/Project3_Design_Document.md and extractor.py.

INSTRUCTION_TEMPLATE = """You are a clinical information extractor. Given a clinical text, extract all
medications and adverse events as a JSON object that follows the schema below.
Return ONLY valid JSON. If no entity is present, return entities=[] and
relation_status="none".

Return ONLY this JSON structure (no record_id, no validation block — those are added by the system):
{{
  "schema_version": "v1",
  "entities": [
    {{
      "entity_type": "medication" | "adverse_event",
      "mention": "<string>",
      "dosage": "<string>" | null,
      "linked_medication": "<string>" | null,
      "evidence": "<string>",
      "source_span": {{"start_char": <int>, "end_char": <int>}}
    }}
  ],
  "relation_status": "related" | "not_related" | "none"
}}

Clinical text:
{text}"""

print("Instruction template loaded.")
print(f"Template length: {len(INSTRUCTION_TEMPLATE)} chars")

Instruction template loaded.
Template length: 756 chars


## Cell 8 — Convert to chat-format JSONL and save

In [14]:
def row_to_chat(row) -> dict:
    """
    Two-turn chat format for Qwen2.5.
    Assistant turn = schema_version + entities[] + relation_status ONLY.
    record_id and validation are NOT included (system-injected at inference, Design Doc D-35).
    """
    user_content      = INSTRUCTION_TEMPLATE.format(text=row["text"])
    assistant_content = json.dumps(row["target_json"], ensure_ascii=False)
    return {
        "messages": [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ],
        "text_hash":     row["text_hash"],
        "source_config": row["source_config"],
    }


def save_jsonl(df_split: pd.DataFrame, path: Path) -> int:
    count = 0
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df_split.iterrows():
            f.write(json.dumps(row_to_chat(row), ensure_ascii=False) + "\n")
            count += 1
    return count


for split_name in ["train", "val", "test"]:
    df_split = df_all[df_all["split"] == split_name].reset_index(drop=True)
    out_path = OUT_DIR / f"{split_name}.jsonl"
    n = save_jsonl(df_split, out_path)
    print(f"Saved {split_name:5s}: {n:6,} rows → {out_path}")

print("\nAll JSONL files saved.")

Saved train: 19,014 rows → /kaggle/working/data/processed/train.jsonl
Saved val  :  2,375 rows → /kaggle/working/data/processed/val.jsonl
Saved test :  2,369 rows → /kaggle/working/data/processed/test.jsonl

All JSONL files saved.


## Cell 9 — Verify JSONL: spot-check first positive record + first negative record

In [15]:
for split_name in ["train", "val", "test"]:
    path = OUT_DIR / f"{split_name}.jsonl"

    # Find first 'related' record in this split for a meaningful check
    first_positive = None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            target = json.loads(rec["messages"][1]["content"])
            if target.get("relation_status") == "related":
                first_positive = (rec, target)
                break

    print(f"\n{'='*60}")
    print(f"Split: {split_name.upper()}")

    if first_positive:
        rec, target = first_positive
        print(f"  First positive example (relation_status=related):")
        print(f"  source_config  : {rec['source_config']}")
        print(f"  entities count : {len(target.get('entities', []))}")
        for ent in target.get("entities", []):
            print(f"    {ent['entity_type']:15s} mention={ent['mention']!r}  span={ent['source_span']}")
        assert "record_id"  not in target, f"{split_name}: 'record_id' must NOT be in target JSON"
        assert "validation" not in target, f"{split_name}: 'validation' must NOT be in target JSON"
        print(f"  ✓ No record_id or validation in target")
    else:
        print(f"  ✗ WARNING: no 'related' records found in {split_name} split!")

print("\nSpot-check complete.")


Split: TRAIN
  First positive example (relation_status=related):
  source_config  : drug_ade_relation
  entities count : 2
    medication      mention='azithromycin'  span={'start_char': 12, 'end_char': 24}
    adverse_event   mention='ototoxicity'  span={'start_char': 33, 'end_char': 44}
  ✓ No record_id or validation in target

Split: VAL
  First positive example (relation_status=related):
  source_config  : drug_ade_relation
  entities count : 2
    medication      mention='naproxen'  span={'start_char': 58, 'end_char': 66}
    adverse_event   mention='pseudoporphyria'  span={'start_char': 32, 'end_char': 47}
  ✓ No record_id or validation in target

Split: TEST
  First positive example (relation_status=related):
  source_config  : drug_ade_relation
  entities count : 2
    medication      mention='dihydrotachysterol'  span={'start_char': 91, 'end_char': 109}
    adverse_event   mention='increased calcium-release'  span={'start_char': 143, 'end_char': 168}
  ✓ No record_id or valid

## Cell 10 — Save dataset_stats.json

In [16]:
token_lengths = all_texts["approx_tokens"].values

stats = {
    "dataset": "ade-benchmark-corpus/ade_corpus_v2",
    "configs_used": [
        "Ade_corpus_v2_classification",
        "Ade_corpus_v2_drug_ade_relation",
        "Ade_corpus_v2_drug_dosage_relation",
    ],
    "raw_row_counts": {
        "classification":      int(len(df_cls)),
        "drug_ade_relation":    int(len(df_ade)),
        "drug_dosage_relation": int(len(df_dos)),
    },
    "unique_texts_across_all_configs": int(len(all_texts)),
    "text_length_approx_tokens": {
        "mean":   round(float(np.mean(token_lengths)), 1),
        "median": round(float(np.median(token_lengths)), 1),
        "p95":    round(float(np.percentile(token_lengths, 95)), 1),
        "max":    int(np.max(token_lengths)),
    },
    "class_balance_classification": {
        str(k): int(v) for k, v in df_cls["label"].value_counts().items()
    },
    "relation_label_counts_all_rows": {
        label: int(count)
        for label, count in df_all["relation_label"].value_counts().items()
    },
    "split_row_counts": {
        split: int((df_all["split"] == split).sum())
        for split in ["train", "val", "test"]
    },
    "split_unique_hash_counts": {
        "train": len(train_hashes),
        "val":   len(val_hashes),
        "test":  len(test_hashes),
    },
    "duplicate_text_rows": {
        "drug_ade_relation":    int(ade_dup),
        "drug_dosage_relation": int(dos_dup),
    },
    "known_limitations": [
        "Dosage field is best-effort: only ~280 drug_dosage_relation rows (~224 after train split).",
        "PubMed case-report style, not clinical-note style. OOD eval set tests generalization.",
        "Sentence-level examples only. Harmony chunker produces short chunks that match this.",
        "Max seq len 256 is safe: p95 approx token count is well below this threshold.",
    ],
    "split_strategy": "80/10/10 grouped by md5(text) hash. Stratified by relation_label. seed=42.",
    "seed": SEED,
}

stats_path = OUT_DIR / "dataset_stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print(f"Saved: {stats_path}")
print(json.dumps(stats, indent=2))

Saved: /kaggle/working/data/processed/dataset_stats.json
{
  "dataset": "ade-benchmark-corpus/ade_corpus_v2",
  "configs_used": [
    "Ade_corpus_v2_classification",
    "Ade_corpus_v2_drug_ade_relation",
    "Ade_corpus_v2_drug_dosage_relation"
  ],
  "raw_row_counts": {
    "classification": 23516,
    "drug_ade_relation": 6821,
    "drug_dosage_relation": 279
  },
  "unique_texts_across_all_configs": 20896,
  "text_length_approx_tokens": {
    "mean": 17.8,
    "median": 17.0,
    "p95": 33.0,
    "max": 135
  },
  "class_balance_classification": {
    "0": 16695,
    "1": 6821
  },
  "relation_label_counts_all_rows": {
    "not_related": 16695,
    "related": 6784,
    "none": 279
  },
  "split_row_counts": {
    "train": 19014,
    "val": 2375,
    "test": 2369
  },
  "split_unique_hash_counts": {
    "train": 16706,
    "val": 2088,
    "test": 2089
  },
  "duplicate_text_rows": {
    "drug_ade_relation": 3956,
    "drug_dosage_relation": 111
  },
  "known_limitations": [
    "Do

## Cell 11 — Final summary

In [17]:
print("=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)

for split_name in ["train", "val", "test"]:
    path = OUT_DIR / f"{split_name}.jsonl"
    size_mb = path.stat().st_size / 1024 / 1024
    n_rows  = sum(1 for _ in open(path, encoding="utf-8"))
    print(f"  {split_name:5s}.jsonl : {n_rows:6,} rows  ({size_mb:.1f} MB)")

print(f"\n  dataset_stats.json : {OUT_DIR / 'dataset_stats.json'}")
print(f"\nNext step: Upload train.jsonl + val.jsonl to Kaggle as a dataset.")
print("Then run notebooks/p3_02_lora_train.ipynb (Phase 3) and")
print("       notebooks/p3_03_qlora_train.ipynb (Phase 4) in parallel.")

DATA PREPARATION COMPLETE
  train.jsonl : 19,014 rows  (25.6 MB)
  val  .jsonl :  2,375 rows  (3.2 MB)
  test .jsonl :  2,369 rows  (3.2 MB)

  dataset_stats.json : /kaggle/working/data/processed/dataset_stats.json

Next step: Upload train.jsonl + val.jsonl to Kaggle as a dataset.
Then run notebooks/p3_02_lora_train.ipynb (Phase 3) and
       notebooks/p3_03_qlora_train.ipynb (Phase 4) in parallel.
